# Reproducible Detection of Three Pharmaceutical Pill Types with YOLO26n

**Course:** Research Methods and Scientific Integrity  
**Project:** Simplified AI-DOTS technical validation  
**Dataset:** MEDISEG v2 — 3-Pills subset  
**Task:** Multi-class object detection (bounding boxes + pill type)

This Colab notebook implements a compact, reproducible experiment. It downloads the official dataset, verifies its checksum, audits the COCO annotations, creates a deterministic train/validation/test split, converts annotations to Ultralytics YOLO format, trains a lightweight pretrained detector, and evaluates the final model on a held-out test set.

> **Research use only.** The resulting model is a technical prototype and must not be used for clinical medication identification.

## 1. Research question and experiment definition

**Research question:** To what extent can a pretrained deep-learning object detector accurately locate and distinguish three pharmaceutical pill types under the visual conditions represented in MEDISEG?

**Objective:** Train and evaluate a YOLO26n model using the MEDISEG 3-Pills subset.

**Primary metrics:** Precision, recall, mAP@0.50, and mAP@0.50:0.95 on the held-out test set.

**Experimental unit:** One annotated image. The split is fixed at 70% training, 15% validation, and 15% test using seed 42.

**Important limitation:** MEDISEG does not provide a participant or acquisition-session grouping variable. Therefore, the reproducible split is image-level and may not estimate performance on a completely independent acquisition site.

## 2. Colab Pro runtime

Before running the notebook, select **Runtime → Change runtime type → T4 GPU, L4 GPU, A100 GPU, or another available GPU**. Then run the cells in order.

In [ ]:
%pip install -q ultralytics==8.4.102

In [ ]:
import csv
import hashlib
import json
import os
import platform
import random
import shutil
import tarfile
import urllib.request
from collections import Counter, defaultdict
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import torch
import ultralytics
import yaml
from PIL import Image, ImageDraw
from ultralytics import YOLO

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

ROOT = Path('/content/mediseg_pill_detection')
ARCHIVE = ROOT / 'MEDISEG.tar.gz'
RAW_ROOT = ROOT / 'raw'
COCO_ROOT = RAW_ROOT / 'MEDISEG' / '3pills'
YOLO_ROOT = ROOT / 'mediseg_3pills_yolo'
RUNS_ROOT = ROOT / 'runs'

FIGSHARE_URL = 'https://ndownloader.figshare.com/files/52926545'
EXPECTED_MD5 = '64d851d97d85de706e941539d48bbd72'
EXPECTED_BYTES = 439_343_024
DATASET_DOI = '10.25383/city.28574786.v2'

ROOT.mkdir(parents=True, exist_ok=True)

print('Python:', platform.python_version())
print('PyTorch:', torch.__version__)
print('Ultralytics:', ultralytics.__version__)
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
else:
    raise RuntimeError('GPU not detected. Enable a GPU runtime before continuing.')

## 3. Download and verify the official MEDISEG archive

The checksum and byte size are verified before extraction. Existing valid files are reused, which makes the cell safe to rerun.

In [ ]:
def md5sum(path: Path, chunk_size: int = 1024 * 1024) -> str:
    digest = hashlib.md5()
    with path.open('rb') as handle:
        for chunk in iter(lambda: handle.read(chunk_size), b''):
            digest.update(chunk)
    return digest.hexdigest()

def download_with_progress(url: str, destination: Path) -> None:
    def report(block_num, block_size, total_size):
        downloaded = min(block_num * block_size, total_size)
        percent = 100 * downloaded / total_size if total_size > 0 else 0
        print(f'\rDownloaded {downloaded / 1e6:.1f}/{total_size / 1e6:.1f} MB ({percent:.1f}%)', end='')
    urllib.request.urlretrieve(url, destination, reporthook=report)
    print()

archive_is_valid = (
    ARCHIVE.exists()
    and ARCHIVE.stat().st_size == EXPECTED_BYTES
    and md5sum(ARCHIVE) == EXPECTED_MD5
)

if not archive_is_valid:
    if ARCHIVE.exists():
        ARCHIVE.unlink()
    download_with_progress(FIGSHARE_URL, ARCHIVE)

actual_md5 = md5sum(ARCHIVE)
assert ARCHIVE.stat().st_size == EXPECTED_BYTES, 'Unexpected archive size.'
assert actual_md5 == EXPECTED_MD5, 'Checksum mismatch: do not use this file.'
print('Verified archive:', ARCHIVE)
print('Bytes:', ARCHIVE.stat().st_size)
print('MD5:', actual_md5)

In [ ]:
annotation_path = COCO_ROOT / 'annotations.json'

if not annotation_path.exists():
    RAW_ROOT.mkdir(parents=True, exist_ok=True)
    with tarfile.open(ARCHIVE, 'r:gz') as archive:
        try:
            archive.extractall(RAW_ROOT, filter='data')
        except TypeError:  # Compatibility fallback for older Python versions
            archive.extractall(RAW_ROOT)

assert annotation_path.exists(), f'Missing {annotation_path}'
print('Extracted dataset:', COCO_ROOT)

## 4. Audit COCO annotations before modeling

This gate checks file references, IDs, category references, and bounding-box geometry. Training stops if a structural problem is detected.

In [ ]:
with annotation_path.open('r', encoding='utf-8') as handle:
    coco = json.load(handle)

images = coco['images']
annotations = coco['annotations']
categories = coco['categories']
images_dir = COCO_ROOT / 'images'

image_by_id = {item['id']: item for item in images}
category_by_id = {item['id']: item['name'] for item in categories}
actual_files = {path.name for path in images_dir.iterdir() if path.is_file()}
declared_files = {item['file_name'] for item in images}

assert len(image_by_id) == len(images), 'Duplicate image IDs found.'
assert len({ann['id'] for ann in annotations}) == len(annotations), 'Duplicate annotation IDs found.'
assert declared_files == actual_files, 'Declared and extracted image files do not match.'

invalid_boxes = []
for ann in annotations:
    assert ann['image_id'] in image_by_id, f"Unknown image_id: {ann['image_id']}"
    assert ann['category_id'] in category_by_id, f"Unknown category_id: {ann['category_id']}"
    x, y, width, height = ann['bbox']
    image_info = image_by_id[ann['image_id']]
    if (width <= 0 or height <= 0 or x < -1 or y < -1
            or x + width > image_info['width'] + 1
            or y + height > image_info['height'] + 1):
        invalid_boxes.append(ann['id'])

assert not invalid_boxes, f'Invalid bounding boxes: {len(invalid_boxes)}'

class_counts = Counter(category_by_id[ann['category_id']] for ann in annotations)
audit_summary = {
    'images': len(images),
    'annotations': len(annotations),
    'categories': category_by_id,
    'annotations_per_category': dict(class_counts),
    'missing_files': 0,
    'invalid_bounding_boxes': 0,
}
print(json.dumps(audit_summary, indent=2, ensure_ascii=False))

## 5. Deterministic split and COCO-to-YOLO conversion

The same sorted image list and seed always produce the same split. A CSV manifest records every assignment. The test set is never used for training or early stopping.

In [ ]:
if YOLO_ROOT.exists():
    shutil.rmtree(YOLO_ROOT)

for split in ('train', 'val', 'test'):
    (YOLO_ROOT / 'images' / split).mkdir(parents=True, exist_ok=True)
    (YOLO_ROOT / 'labels' / split).mkdir(parents=True, exist_ok=True)

ordered_ids = sorted(image_by_id)
rng = random.Random(SEED)
rng.shuffle(ordered_ids)

n_images = len(ordered_ids)
n_train = round(n_images * 0.70)
n_val = round(n_images * 0.15)
split_ids = {
    'train': ordered_ids[:n_train],
    'val': ordered_ids[n_train:n_train + n_val],
    'test': ordered_ids[n_train + n_val:],
}

category_ids = sorted(category_by_id)
category_to_yolo = {category_id: index for index, category_id in enumerate(category_ids)}
class_names = [category_by_id[category_id] for category_id in category_ids]
annotations_by_image = defaultdict(list)
for ann in annotations:
    annotations_by_image[ann['image_id']].append(ann)

manifest_rows = []
split_class_counts = {}
for split, ids in split_ids.items():
    counts = Counter()
    for image_id in ids:
        info = image_by_id[image_id]
        source_path = images_dir / info['file_name']
        target_image = YOLO_ROOT / 'images' / split / info['file_name']
        shutil.copy2(source_path, target_image)

        label_path = YOLO_ROOT / 'labels' / split / f"{Path(info['file_name']).stem}.txt"
        label_lines = []
        for ann in annotations_by_image[image_id]:
            x, y, width, height = ann['bbox']
            image_width, image_height = info['width'], info['height']
            x_center = (x + width / 2) / image_width
            y_center = (y + height / 2) / image_height
            norm_width = width / image_width
            norm_height = height / image_height
            class_id = category_to_yolo[ann['category_id']]
            values = [x_center, y_center, norm_width, norm_height]
            assert all(-1e-6 <= value <= 1 + 1e-6 for value in values)
            label_lines.append(
                f'{class_id} {x_center:.8f} {y_center:.8f} {norm_width:.8f} {norm_height:.8f}'
            )
            counts[class_names[class_id]] += 1
        label_path.write_text('\n'.join(label_lines) + ('\n' if label_lines else ''), encoding='utf-8')
        manifest_rows.append({'image_id': image_id, 'file_name': info['file_name'], 'split': split})
    split_class_counts[split] = dict(counts)

manifest_path = YOLO_ROOT / 'split_manifest.csv'
with manifest_path.open('w', newline='', encoding='utf-8') as handle:
    writer = csv.DictWriter(handle, fieldnames=['image_id', 'file_name', 'split'])
    writer.writeheader()
    writer.writerows(manifest_rows)

data_yaml = {
    'path': str(YOLO_ROOT),
    'train': 'images/train',
    'val': 'images/val',
    'test': 'images/test',
    'names': {index: name for index, name in enumerate(class_names)},
}
data_yaml_path = YOLO_ROOT / 'data.yaml'
with data_yaml_path.open('w', encoding='utf-8') as handle:
    yaml.safe_dump(data_yaml, handle, sort_keys=False, allow_unicode=True)

dataset_summary = {
    'dataset': 'MEDISEG v2 - 3pills',
    'doi': DATASET_DOI,
    'archive_md5': EXPECTED_MD5,
    'seed': SEED,
    'split_images': {key: len(value) for key, value in split_ids.items()},
    'split_annotations_by_class': split_class_counts,
    'class_names': class_names,
}
(YOLO_ROOT / 'dataset_summary.json').write_text(
    json.dumps(dataset_summary, indent=2, ensure_ascii=False), encoding='utf-8'
)

for split in ('train', 'val', 'test'):
    assert len(list((YOLO_ROOT / 'images' / split).glob('*'))) == len(split_ids[split])
    assert len(list((YOLO_ROOT / 'labels' / split).glob('*.txt'))) == len(split_ids[split])
    assert all(split_class_counts[split].get(name, 0) > 0 for name in class_names)

print(json.dumps(dataset_summary, indent=2, ensure_ascii=False))
print('YOLO configuration:', data_yaml_path)
print('Split manifest:', manifest_path)

In [ ]:
def show_yolo_samples(split='train', count=6):
    image_paths = sorted((YOLO_ROOT / 'images' / split).glob('*'))
    sample_rng = random.Random(SEED)
    selected = sample_rng.sample(image_paths, min(count, len(image_paths)))
    fig, axes = plt.subplots(2, 3, figsize=(15, 10))
    for axis, image_path in zip(axes.ravel(), selected):
        image = Image.open(image_path).convert('RGB')
        draw = ImageDraw.Draw(image)
        width, height = image.size
        label_path = YOLO_ROOT / 'labels' / split / f'{image_path.stem}.txt'
        for line in label_path.read_text(encoding='utf-8').splitlines():
            class_id, xc, yc, bw, bh = map(float, line.split())
            x1 = (xc - bw / 2) * width
            y1 = (yc - bh / 2) * height
            x2 = (xc + bw / 2) * width
            y2 = (yc + bh / 2) * height
            draw.rectangle((x1, y1, x2, y2), outline='red', width=3)
            draw.text((x1, max(0, y1 - 14)), class_names[int(class_id)], fill='red')
        axis.imshow(image)
        axis.set_title(image_path.name[:35])
        axis.axis('off')
    for axis in axes.ravel()[len(selected):]:
        axis.axis('off')
    plt.tight_layout()

show_yolo_samples('train', 6)

## 6. Train YOLO26n

The nano checkpoint is selected to obtain a defensible baseline within the course deadline. Training uses fixed software, model, seed, image size, and output paths. Early stopping monitors validation performance; the test set remains untouched.

In [ ]:
model = YOLO('yolo26n.pt')
train_results = model.train(
    data=str(data_yaml_path),
    epochs=40,
    patience=8,
    imgsz=640,
    batch=16,
    device=0,
    workers=2,
    seed=SEED,
    deterministic=True,
    pretrained=True,
    cache='disk',
    plots=True,
    save=True,
    project=str(RUNS_ROOT),
    name='yolo26n_mediseg3',
    exist_ok=True,
    verbose=True,
)

best_weights = RUNS_ROOT / 'yolo26n_mediseg3' / 'weights' / 'best.pt'
assert best_weights.exists(), 'Training finished without best.pt.'
print('Best weights:', best_weights)

## 7. Final evaluation on the held-out test set

This is the only cell that uses the test split. Report these values as the final technical results, without interpreting them as clinical effectiveness.

In [ ]:
best_model = YOLO(str(best_weights))
test_metrics = best_model.val(
    data=str(data_yaml_path),
    split='test',
    imgsz=640,
    batch=16,
    device=0,
    plots=True,
    project=str(RUNS_ROOT),
    name='test_evaluation',
)

metric_summary = {
    'precision': float(test_metrics.box.mp),
    'recall': float(test_metrics.box.mr),
    'mAP50': float(test_metrics.box.map50),
    'mAP50_95': float(test_metrics.box.map),
    'per_class_mAP50_95': {
        class_names[index]: float(value)
        for index, value in enumerate(test_metrics.box.maps)
    },
    'model': 'yolo26n.pt',
    'ultralytics_version': ultralytics.__version__,
    'torch_version': torch.__version__,
    'gpu': torch.cuda.get_device_name(0),
    'seed': SEED,
    'dataset_doi': DATASET_DOI,
    'dataset_md5': EXPECTED_MD5,
    'test_images': len(split_ids['test']),
}
metrics_path = RUNS_ROOT / 'test_metrics.json'
metrics_path.write_text(json.dumps(metric_summary, indent=2), encoding='utf-8')
print(json.dumps(metric_summary, indent=2))

In [ ]:
test_images = sorted((YOLO_ROOT / 'images' / 'test').glob('*'))
prediction_rng = random.Random(SEED)
prediction_sample = prediction_rng.sample(test_images, min(9, len(test_images)))
prediction_results = best_model.predict(
    source=[str(path) for path in prediction_sample],
    imgsz=640,
    conf=0.25,
    device=0,
    save=True,
    project=str(RUNS_ROOT),
    name='test_predictions',
    exist_ok=True,
)

fig, axes = plt.subplots(3, 3, figsize=(15, 15))
for axis, result in zip(axes.ravel(), prediction_results):
    rendered = result.plot()[:, :, ::-1]
    axis.imshow(rendered)
    axis.axis('off')
plt.tight_layout()

## 8. Package reproducibility artifacts

The ZIP includes weights, training curves, confusion matrices, test metrics, predictions, split manifest, dataset summary, and YAML configuration. Download it before closing the Colab runtime.

In [ ]:
artifact_root = ROOT / 'submission_artifacts'
if artifact_root.exists():
    shutil.rmtree(artifact_root)
artifact_root.mkdir(parents=True)

shutil.copytree(RUNS_ROOT, artifact_root / 'runs')
for path in (
    YOLO_ROOT / 'data.yaml',
    YOLO_ROOT / 'split_manifest.csv',
    YOLO_ROOT / 'dataset_summary.json',
):
    shutil.copy2(path, artifact_root / path.name)

environment = {
    'python': platform.python_version(),
    'torch': torch.__version__,
    'ultralytics': ultralytics.__version__,
    'gpu': torch.cuda.get_device_name(0),
}
(artifact_root / 'environment.json').write_text(
    json.dumps(environment, indent=2), encoding='utf-8'
)

zip_path = Path(shutil.make_archive('/content/MEDISEG_YOLO26_results', 'zip', artifact_root))
print(f'Created {zip_path} ({zip_path.stat().st_size / 1e6:.1f} MB)')

In [ ]:
from google.colab import files
files.download(str(zip_path))

### Optional backup to Google Drive

If the browser download is interrupted, run the next cell and authorize Drive.

In [ ]:
# Optional: uncomment and run to preserve the ZIP in Google Drive.
# from google.colab import drive
# drive.mount('/content/drive')
# destination = Path('/content/drive/MyDrive/AI_DOTS/MEDISEG_YOLO26_results.zip')
# destination.parent.mkdir(parents=True, exist_ok=True)
# shutil.copy2(zip_path, destination)
# print('Saved:', destination)

## 9. Result statement template

Complete this statement only after the test-evaluation cell has run:

> On the held-out MEDISEG 3-Pills test set (n = **[test images]**), the YOLO26n detector obtained precision **[value]**, recall **[value]**, mAP@0.50 **[value]**, and mAP@0.50:0.95 **[value]**. These findings establish technical performance within this dataset and do not constitute clinical validation.